# 05B Global Model Transfer Only

This notebook runs a strict saved-artifact transfer evaluation on M5 using synthetic-trained global boosting artifacts.

Important:
- this is an honest saved-model transfer test
- it does **not** retrain on M5
- it produces transfer metrics on monthly aggregated M5 data
- it is not the same as a Kaggle daily submission, because the saved synthetic artifacts are monthly models. This notebook is transfer evaluation only and does not generate uploadable XGBOOST/CATBOOST Kaggle CSVs.
- prerequisite: saved artifacts must already exist under `modeling/outputs/artifacts` (for example from notebook 02 global model training and artifact save)

In [22]:
from pathlib import Path
import pandas as pd

M5_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy')
REPORTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')
SCRIPT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py')
TAG = 'strict_m5_transfer'

def resolve_report_path(kind: str, preferred_tag: str = TAG) -> Path:
    if not REPORTS_DIR.exists():
        raise FileNotFoundError(
            f'Reports directory not found: {REPORTS_DIR}. Run the transfer cell above first.'
        )

    preferred = REPORTS_DIR / f'{preferred_tag}_{kind}.csv'
    if preferred.exists():
        return preferred

    available = sorted(p.name for p in REPORTS_DIR.glob(f'{preferred_tag}_*.csv'))
    raise FileNotFoundError(
        f'Missing transfer report: {preferred.name}. Run the transfer cell above first. '
        f'Available transfer CSVs for tag {preferred_tag!r}: {available}'
    )

## Run Transfer Evaluation

In [23]:
ARTIFACTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/artifacts')
required_checks = [
    ARTIFACTS_DIR / 'A' / 'xgboost_h1' / 'production' / 'metadata.json',
    ARTIFACTS_DIR / 'A' / 'catboost_h1' / 'production' / 'metadata.json',
]
missing = [str(p) for p in required_checks if not p.exists()]

if missing:
    raise FileNotFoundError(
        'Missing saved model artifacts required for transfer evaluation. '
        'Run notebook 02_global_model_training_and_artifact_save.ipynb first. '
        f'Missing examples: {missing}'
    )

!python "{SCRIPT}" --m5-dir "{M5_DIR}" --granularity dept_store --datasets A B C --models XGBOOST CATBOOST --tag "{TAG}"

Saved predictions: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/strict_m5_transfer_predictions.csv
Saved metrics: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/strict_m5_transfer_metrics.csv
Saved summary: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/strict_m5_transfer_summary.csv


## Summary

In [24]:
!python "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py" \
  --m5-dir "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy" \
  --granularity dept_store \
  --datasets B \
  --models XGBOOST CATBOOST \
  --tag "strict_m5_transfer"


Saved predictions: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/strict_m5_transfer_predictions.csv
Saved metrics: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/strict_m5_transfer_metrics.csv
Saved summary: /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports/strict_m5_transfer_summary.csv


In [25]:
summary_path = resolve_report_path('summary')
pd.read_csv(summary_path)


,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_B,XGBOOST,test,0,10080,0.174385,4692.680489,-643.308256,6.318099,1476.385674,0.529563
1,M5_TRANSFER_B,CATBOOST,test,0,10080,0.260936,7172.979802,-1115.275120,9.848071,2209.154301,0.485119


## Detailed Horizon Metrics

In [26]:
metrics_path = resolve_report_path('metrics')
pd.read_csv(metrics_path).head(50)

,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_B,XGBOOST,test,1,840,0.163388,4203.992227,192.351783,6.700036,1383.286401,0.416667
1,M5_TRANSFER_B,XGBOOST,test,2,840,0.184906,5171.980202,-1001.001714,6.647497,1565.459876,0.554762
2,M5_TRANSFER_B,XGBOOST,test,3,840,0.162167,4334.679105,79.028646,6.033561,1372.949917,0.417857
3,M5_TRANSFER_B,XGBOOST,test,4,840,0.169658,4601.031872,-458.466492,5.865846,1436.369836,0.459524
4,M5_TRANSFER_B,XGBOOST,test,5,840,0.169177,4584.728092,-620.999174,5.668903,1432.294922,0.513095
5,M5_TRANSFER_B,XGBOOST,test,6,840,0.171767,4519.186494,-423.879315,6.818633,1454.224741,0.498810
6,M5_TRANSFER_B,XGBOOST,test,7,840,0.181420,4521.149631,-876.521676,6.575810,1535.949745,0.597619
7,M5_TRANSFER_B,XGBOOST,test,8,840,0.170725,4480.480406,-360.335826,6.812662,1445.399802,0.536905
8,M5_TRANSFER_B,XGBOOST,test,9,840,0.160104,4413.586178,-419.068852,6.212695,1355.480550,0.495238
9,M5_TRANSFER_B,XGBOOST,test,10,840,0.172806,4514.996651,-987.668331,5.962534,1463.023676,0.603571


## Why This Is Not a Kaggle Submission

The saved artifacts are monthly synthetic-trained models. Kaggle M5 submission requires 28-day daily item-store forecasts. That means:
- this notebook is valid for transfer evaluation
- it is not valid for strict Kaggle submission generation from the same saved monthly artifacts